In [ ]:
def train_fashion_mnist():
    import importlib.util
    import pathlib
    import sys

    script_path = pathlib.Path("/etc/mpi-test/fashion_mnist_mpi.py")
    if not script_path.exists():
        raise RuntimeError(f"Expected mounted MPI script at {script_path}")

    module_name = "fashion_mnist_mpi"
    spec = importlib.util.spec_from_file_location(module_name, script_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Unable to load module from {script_path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    module.train_mpi_fashion_mnist()


In [ ]:
import os

from kubernetes import client as k8s_client
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient

openshift_api_url = os.getenv("OPENSHIFT_API_URL", "")
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not openshift_api_url or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN are required")

cfg = k8s_client.Configuration()
cfg.host = openshift_api_url
ca_bundle = os.getenv("OPENSHIFT_CA_BUNDLE", "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt")
if os.path.exists(ca_bundle):
    cfg.ssl_ca_cert = ca_bundle
    cfg.verify_ssl = True
elif os.getenv("NOTEBOOK_INSECURE_TLS") == "1":
    cfg.verify_ssl = False
else:
    raise RuntimeError(
        f"No CA bundle found at {ca_bundle}; set OPENSHIFT_CA_BUNDLE or NOTEBOOK_INSECURE_TLS=1 to opt out"
    )
cfg.api_key = {"authorization": f"Bearer {token}"}

api_client = k8s_client.ApiClient(cfg)
backend_cfg = KubernetesBackendConfig(
    client_configuration=api_client.configuration,
)
client = TrainerClient(backend_cfg)


In [ ]:
training_runtime_name = os.getenv("TRAINING_RUNTIME", "")
if not training_runtime_name:
    raise RuntimeError("TRAINING_RUNTIME environment variable is required")

try:
    mpi_runtime = client.get_runtime(training_runtime_name)
except Exception as e:
    raise RuntimeError(f"Runtime '{training_runtime_name}' not found or not accessible") from e

kueue_queue_name = os.getenv("KUEUE_QUEUE_NAME", "")
if kueue_queue_name:
    print(f"[notebook] Kueue enabled with LocalQueue: {kueue_queue_name}")
else:
    print("[notebook] KUEUE_QUEUE_NAME not set; TrainJob will not request an explicit queue")


In [ ]:
import os

from kubeflow.trainer import CustomTrainer
from kubeflow.trainer.options import (
    ContainerOverride,
    Labels,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)

num_nodes = 2
script_mount_path = "/etc/mpi-test"
notebook_configmap_name = os.getenv("NOTEBOOK_CONFIGMAP_NAME", "")
if not notebook_configmap_name:
    raise RuntimeError("NOTEBOOK_CONFIGMAP_NAME environment variable is required")

train_options = [
    PodTemplateOverrides(
        PodTemplateOverride(
            target_jobs=["launcher", "node"],
            spec=PodSpecOverride(
                volumes=[
                    {
                        "name": "mpi-test-script",
                        "configMap": {"name": notebook_configmap_name},
                    }
                ],
                containers=[
                    ContainerOverride(
                        name="node",
                        volume_mounts=[
                            {
                                "name": "mpi-test-script",
                                "mountPath": script_mount_path,
                                "readOnly": True,
                            }
                        ],
                    )
                ],
            ),
        )
    )
]
if kueue_queue_name:
    train_options.append(Labels(labels={"kueue.x-k8s.io/queue-name": kueue_queue_name}))

job_name = client.train(
    trainer=CustomTrainer(
        func=train_fashion_mnist,
        num_nodes=num_nodes,
        resources_per_node={
            "cpu": 2,
            "memory": "8Gi",
            "nvidia.com/gpu": 1,
        },
    ),
    runtime=mpi_runtime,
    options=train_options,
)

print(f"[notebook] Job submitted: {job_name}")


In [ ]:
client.wait_for_job_status(name=job_name, status={"Running", "Complete", "Failed"}, timeout=300)
client.wait_for_job_status(name=job_name, status={"Complete", "Failed"}, timeout=600)

job = client.get_job(name=job_name)
logs = list(client.get_job_logs(name=job_name, follow=False))
log_text = "\n".join(str(line) for line in logs)

print(f"Training job final status: {job.status}")

if job.status == "Failed":
    print("ERROR: MPI SDK training job failed")
    for line in logs[-50:]:
        print(line)
    raise RuntimeError(f"MPI SDK training job '{job_name}' failed")

if "MPI TrainJob test PASSED" not in log_text:
    print("ERROR: MPI success marker not found in logs")
    for line in logs[-80:]:
        print(line)
    raise RuntimeError("MPI SDK training did not complete successfully")

print(f"✓ MPI SDK training job '{job_name}' completed successfully")
